# Discrete Cosine Transform (DCT)

Import Library

In [24]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.fftpack import dct, idct
import ipywidgets as widgets
from ipywidgets import interact

In [25]:
nama_folder = 'Pict'

if not os.path.exists(nama_folder):
    os.makedirs(nama_folder)
    print(f"Folder '{nama_folder}' dibuat.")

# Ekstensi yang didukung
ekstensi_valid = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
daftar_gambar = [f for f in os.listdir(nama_folder) if f.lower().endswith(ekstensi_valid)]

if not daftar_gambar:
    print(f"Belum ada gambar di folder '{nama_folder}'.")
else:
    print(f"Ditemukan {len(daftar_gambar)} gambar siap uji.")

Ditemukan 3 gambar siap uji.


Fungsi Matematika DCT dan IDCT

In [26]:
def dct2(block):
    # Transformasi DCT pada baris, lalu pada kolom (transpose)
    return dct(dct(block.T, norm='ortho').T, norm='ortho')

def idct2(block):
    # Transformasi Inverse DCT untuk mengembalikan ke nilai piksel
    return idct(idct(block.T, norm='ortho').T, norm='ortho')

Pemrosesan, Visualisasi, dan Analisis Error

In [27]:
def proses_dan_tampilkan_dct(nama_file, simulasi_kompresi=False):
    path_lengkap = os.path.join(nama_folder, nama_file)
    
    # Baca gambar dalam Grayscale
    img = cv2.imread(path_lengkap, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"Gagal memuat gambar: {path_lengkap}")
        return
        
    # ukuran kelipatan 8
    h, w = img.shape
    h = h - (h % 8)
    w = w - (w % 8)
    img = img[:h, :w]
    
    # Matriks kosong
    dct_img = np.zeros((h, w), dtype=np.float32)
    reconstructed_img = np.zeros((h, w), dtype=np.float32)
    
    # Perulangan pemrosesan matriks blok 8x8
    for i in range(0, h, 8):
        for j in range(0, w, 8):
            block = img[i:i+8, j:j+8]
            dct_block = dct2(block)
            
            # Simulasi Kompresi (Membuang frekuensi tinggi / detail kecil)
            if simulasi_kompresi:
                dct_block[4:, :] = 0
                dct_block[:, 4:] = 0
                
            dct_img[i:i+8, j:j+8] = dct_block
            reconstructed_img[i:i+8, j:j+8] = idct2(dct_block)
            
    reconstructed_img = np.clip(reconstructed_img, 0, 255)
    
    # Analisis error
    # Hitung Persentase Data/Detail yang Dibuang (Angka 0 pada DCT)
    total_koefisien = h * w
    koefisien_nol = np.sum(dct_img == 0)
    persentase_hilang = (koefisien_nol / total_koefisien) * 100
    
    # Hitung Mean Squared Error (MSE) antara Piksel Asli vs Rekonstruksi
    mse = np.mean((img.astype(np.float32) - reconstructed_img) ** 2)
    
    # Tampilkan Teks Metrik
    print("="*40)
    print(f"Nama file                  : {nama_file}")
    print(f"Persentase Detail Dibuang   : {persentase_hilang:.2f}%")
    print(f"Tingkat Error Piksel (MSE)  : {mse:.2f}")
    
    if mse < 1:
        print("Status Kualitas             : Sangat Baik (Lossless / Identik)")
    else:
        print("Status Kualitas             : Menurun (Lossy Compression)")
    print("="*40)
    
    # Visual
    plt.figure(figsize=(18, 6))
    
    plt.subplot(1, 3, 1)
    plt.imshow(img, cmap='gray')
    plt.title(f'1. Citra Asli\n({nama_file})')
    plt.axis('off')
    
    plt.subplot(1, 3, 2)
    plt.imshow(np.log(np.abs(dct_img) + 1), cmap='gray')
    plt.title('2. Koefisien DCT\n(Domain Frekuensi)')
    plt.axis('off')
    
    plt.subplot(1, 3, 3)
    plt.imshow(reconstructed_img, cmap='gray')
    plt.title(f'3. Hasil Rekonstruksi\nMSE: {mse:.2f}')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

In [29]:
if daftar_gambar:
    interact(proses_dan_tampilkan_dct, 
             nama_file=daftar_gambar, 
             simulasi_kompresi=False)

interactive(children=(Dropdown(description='nama_file', options=('Logo Softdev.jpeg', 'test.jpg', 'WhatsApp Im…